In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from google.cloud import bigquery
import pandas as pd

project_id = "YOUR_PROJECT_ID"

client = bigquery.Client(project=project_id)

query = """
SELECT *
FROM `YOUR_PROJECT_ID.macro_ml_project.macro_stress_ready`
"""

df = client.query(query).to_dataframe()

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
## df.to_csv("macro_stress_ready.csv", index=False)

In [ ]:
## from google.colab import files
## files.download("macro_stress_ready.csv")

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

In [ ]:
df.head()

df.shape

In [ ]:
df.shape
df.info()
df.isnull().sum().sort_values(ascending=False)
df.describe().T

In [ ]:
df["economic_stress"].value_counts()

In [ ]:
df["economic_stress"].value_counts(normalize=True) * 100

In [ ]:
sns.countplot(data=df, x="economic_stress")
plt.title("Class Distribution of Economic Stress")
plt.xlabel("Economic Stress")
plt.ylabel("Count")
plt.show()

In [ ]:
## Missing values checker
missing_pct = df.isnull().mean().sort_values(ascending=False) * 100
missing_pct

In [ ]:
## drop value if missing pct
df = df.dropna(subset=["economic_stress"])

In [ ]:
## stress rule definition
stress_rules = {
    "Negative GDP growth": df["gdp_growth_pct"] < 0,
    "High inflation": df["inflation_pct"] > 10,
    "High unemployment": df["unemployment_pct"] > 10,
    "Large current account deficit": df["current_account_pct_gdp"] < -5,
    "Large exchange rate depreciation": df["exchange_rate_change_pct"] > 20
}

In [ ]:
## define features and target
target = "economic_stress"

feature_cols = [
    "gdp_growth_pct",
    "gdp_per_capita_current_usd",
    "inflation_pct",
    "unemployment_pct",
    "current_account_pct_gdp",
    "exports_pct_gdp",
    "imports_pct_gdp",
    "trade_pct_gdp",
    "fdi_net_inflows_pct_gdp",
    "exchange_rate_lcu_per_usd",
    "population",
    "misery_index",
    "trade_balance_proxy",
    "growth_inflation_gap",
    "external_pressure_score",
    "exchange_rate_change_pct"
]

X = df[feature_cols]
y = df[target]

X.head()


In [ ]:
y.head()

In [ ]:
## summary statistic stress class

df.groupby("economic_stress")[feature_cols].mean().T

In [ ]:
summary_by_class = df.groupby("economic_stress")[feature_cols].agg(["mean", "median", "std"])
summary_by_class

In [ ]:
## Variable roles
variable_role_table = pd.DataFrame({
    "Variable Group": [
        "Identifier variables",
        "Target variable",
        "Core stress-definition indicators",
        "Additional macroeconomic predictors",
        "Engineered predictors"
    ],
    "Examples": [
        "country_name, country_code, year",
        "economic_stress",
        "gdp_growth_pct, inflation_pct, unemployment_pct, current_account_pct_gdp, exchange_rate_change_pct",
        "gdp_per_capita_current_usd, exports_pct_gdp, imports_pct_gdp, trade_pct_gdp, fdi_net_inflows_pct_gdp, exchange_rate_lcu_per_usd, population",
        "misery_index, trade_balance_proxy, growth_inflation_gap, external_pressure_score"
    ],
    "Role in Analysis": [
        "Identify each country-year observation; not used as model predictors.",
        "Binary label to be predicted.",
        "Used to define whether a country-year is classified as stressed.",
        "Provide broader economic context for classification.",
        "Summarize macroeconomic pressure and support model interpretation."
    ]
})

variable_role_table

In [ ]:
## Histogram
for col in [
    "gdp_growth_pct",
    "inflation_pct",
    "unemployment_pct",
    "current_account_pct_gdp",
    "exchange_rate_change_pct"
]:
    plt.figure(figsize=(7, 5))
    sns.histplot(data=df, x=col, hue="economic_stress", kde=True, bins=30)
    plt.title(f"Distribution of {col} by Economic Stress")
    plt.show()

In [ ]:
stress_indicator_vars = [
    "gdp_growth_pct",
    "inflation_pct",
    "unemployment_pct",
    "current_account_pct_gdp",
    "exchange_rate_change_pct"
]

stress_indicator_labels = {
    "gdp_growth_pct": "GDP Growth (%)",
    "inflation_pct": "Inflation (%)",
    "unemployment_pct": "Unemployment (%)",
    "current_account_pct_gdp": "Current Account (% of GDP)",
    "exchange_rate_change_pct": "Exchange Rate Change (%)"
}

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()

for i, var in enumerate(stress_indicator_vars):
    sns.boxplot(
        data=df,
        x="economic_stress",
        y=var,
        ax=axes[i]
    )
    axes[i].set_title(stress_indicator_labels[var] + " by Stress Class")
    axes[i].set_xlabel("Economic Stress")
    axes[i].set_ylabel(stress_indicator_labels[var])
    axes[i].set_xticklabels(["No Stress", "Stress"])

# Remove empty sixth subplot
fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

In [ ]:
## correlation matrix
plt.figure(figsize=(14, 10))
corr = df[feature_cols + ["economic_stress"]].corr()
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Macroeconomic Indicators")
plt.show()

In [ ]:

# CREATE LAGGED FEATURES FOR EARLY WARNING SETUP


df_lagged = df.sort_values(["country_code", "year"]).copy()

base_features = [
    "gdp_growth_pct",
    "gdp_per_capita_current_usd",
    "inflation_pct",
    "unemployment_pct",
    "current_account_pct_gdp",
    "exports_pct_gdp",
    "imports_pct_gdp",
    "trade_pct_gdp",
    "fdi_net_inflows_pct_gdp",
    "exchange_rate_lcu_per_usd",
    "population",
    "misery_index",
    "trade_balance_proxy",
    "growth_inflation_gap",
    "external_pressure_score",
    "exchange_rate_change_pct"
]

for col in base_features:
    df_lagged[col + "_lag1"] = df_lagged.groupby("country_code")[col].shift(1)

lagged_feature_cols = [col + "_lag1" for col in base_features]

df_lagged_model = df_lagged.dropna(subset=lagged_feature_cols + ["economic_stress"]).copy()

X_lagged = df_lagged_model[lagged_feature_cols]
y_lagged = df_lagged_model["economic_stress"]

print("Lagged model dataset shape:", df_lagged_model.shape)
print("Number of lagged features:", len(lagged_feature_cols))

In [ ]:
feature_cols = [
    "gdp_growth_pct",
    "gdp_per_capita_current_usd",
    "inflation_pct",
    "unemployment_pct",
    "current_account_pct_gdp",
    "exports_pct_gdp",
    "imports_pct_gdp",
    "trade_pct_gdp",
    "fdi_net_inflows_pct_gdp",
    "exchange_rate_lcu_per_usd",
    "population",
    "misery_index",
    "trade_balance_proxy",
    "growth_inflation_gap",
    "external_pressure_score",
    "exchange_rate_change_pct"
]

X = df[feature_cols]
y = df["economic_stress"]

In [ ]:
corr_with_target = corr["economic_stress"].sort_values(ascending=False)
corr_with_target

In [ ]:
## train split test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
y_train.value_counts(normalize=True)

In [ ]:
y_test.value_counts(normalize=True)

In [ ]:
# TIME BASED TRAIN TEST SPLIT
# Train on earlier years, test on later years


train_data = df_lagged_model[df_lagged_model["year"] <= 2016]
test_data = df_lagged_model[df_lagged_model["year"] > 2016]

X_train_time = train_data[lagged_feature_cols]
y_train_time = train_data["economic_stress"]

X_test_time = test_data[lagged_feature_cols]
y_test_time = test_data["economic_stress"]

print("Training period:", train_data["year"].min(), "-", train_data["year"].max())
print("Testing period:", test_data["year"].min(), "-", test_data["year"].max())
print("Train shape:", X_train_time.shape)
print("Test shape:", X_test_time.shape)

In [ ]:
## Preprocessing

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

logistic_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

tree_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

In [ ]:
## set up model
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocess", logistic_preprocess),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Decision Tree": Pipeline(steps=[
        ("preprocess", tree_preprocess),
        ("model", DecisionTreeClassifier(
            max_depth=5,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocess", tree_preprocess),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Gradient Boosting": Pipeline(steps=[
        ("preprocess", tree_preprocess),
        ("model", GradientBoostingClassifier(
            random_state=42
        ))
    ])
}

In [ ]:
## processing pipeline
## Missing value imputation.
## Scaling for Logistic Regression.
##Tree models do not strictly need scaling, but using one pipeline keeps things neat
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

numeric_features = feature_cols

numeric_transformer_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_transformer_unscaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

logistic_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

tree_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

In [ ]:
## Log regression
log_reg_model = Pipeline(steps=[
    ("preprocessor", numeric_transformer_scaled),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

In [ ]:
## decision tree
decision_tree_model = Pipeline(steps=[
    ("preprocessor", numeric_transformer_unscaled),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
## random forest

random_forest_model = Pipeline(steps=[
    ("preprocessor", numeric_transformer_unscaled),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
## gradient boosting
gradient_boosting_model = Pipeline(steps=[
    ("preprocessor", numeric_transformer_unscaled),
    ("classifier", GradientBoostingClassifier(random_state=42))
])

In [ ]:
## train and evaluation
## Evaluate
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    if hasattr(model.named_steps["classifier"], "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)
    else:
        auc = np.nan

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": auc
    }

    print(f"\n===== {model_name} =====")
    print(classification_report(y_test, y_pred))

    ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

    return results

In [ ]:
models = [
    (log_reg_model, "Logistic Regression"),
    (decision_tree_model, "Decision Tree"),
    (random_forest_model, "Random Forest"),
    (gradient_boosting_model, "Gradient Boosting")
]

results = []

for model, name in models:
    result = evaluate_model(model, X_train, X_test, y_train, y_test, name)
    results.append(result)

results_df = pd.DataFrame(results)
results_df

In [ ]:
from sklearn.dummy import DummyClassifier

baseline_model = Pipeline(steps=[
    ("preprocessor", numeric_transformer_unscaled),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

baseline_result = evaluate_model(
    baseline_model,
    X_train,
    X_test,
    y_train,
    y_test,
    "Baseline - Most Frequent Class"
)

baseline_result

In [ ]:
results_df.sort_values(by="F1 Score", ascending=False)

In [ ]:
results_df.sort_values(by="Recall", ascending=False)

In [ ]:
##cross validation to robust result
for model, name in models:
    cv_scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="f1"
    )

    print(name)
    print("Mean F1:", cv_scores.mean())
    print("Std F1:", cv_scores.std())
    print()

In [ ]:
##recall validation
for model, name in models:
    cv_scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="recall"
    )

    print(name)
    print("Mean Recall:", cv_scores.mean())
    print("Std Recall:", cv_scores.std())
    print()

In [ ]:
##ROC

fig, ax = plt.subplots(figsize=(8, 6))

for model, name in models:
    model.fit(X_train, y_train)
    RocCurveDisplay.from_estimator(
        model,
        X_test,
        y_test,
        name=name,
        ax=ax
    )

plt.title("ROC Curves for Macroeconomic Stress Models")
plt.show()

In [ ]:
## FEATURE IMPORTANCE
random_forest_model.fit(X_train, y_train)

In [ ]:
## extract and plot
rf_classifier = random_forest_model.named_steps["classifier"]

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_classifier.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x="importance", y="feature")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

In [ ]:
## model comparison chart

results_long = results_df.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=results_long,
    x="Model",
    y="Score",
    hue="Metric"
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xlabel("Model")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.legend(title="Metric", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
## decision tree viz
decision_tree_model.fit(X_train, y_train)

tree_classifier = decision_tree_model.named_steps["classifier"]

plt.figure(figsize=(22, 10))
plot_tree(
    tree_classifier,
    feature_names=feature_cols,
    class_names=["Normal", "Stress"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Decision Tree Rules for Macroeconomic Stress")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### DEFINE MACROECONOMIC STRESS RULES

# Adjust these thresholds if your BigQuery rule used different values.
# This version follows a simple early-warning logic:
# A country-year is stressed if at least one macroeconomic vulnerability condition is met.

stress_rules = {
    "Negative GDP growth": df["gdp_growth_pct"] < 0,
    "High inflation": df["inflation_pct"] > 10,
    "High unemployment": df["unemployment_pct"] > 10,
    "Large current account deficit": df["current_account_pct_gdp"] < -5,
    "Large exchange rate depreciation": df["exchange_rate_change_pct"] > 20
}

# Create rule flag columns: 1 if condition is met, 0 otherwise
for rule_name, condition in stress_rules.items():
    df[rule_name] = condition.astype(int)

# Count how many stress conditions are met by each country-year
rule_cols = list(stress_rules.keys())
df["number_of_stress_conditions"] = df[rule_cols].sum(axis=1)

# Reconstruct rule-based stress label
df["economic_stress_rule_based"] = np.where(
    df["number_of_stress_conditions"] >= 1,
    1,
    0
)

# Check whether this recreated label matches your existing target variable
if "economic_stress" in df.columns:
    match_rate = (df["economic_stress"] == df["economic_stress_rule_based"]).mean()
    print(f"Match rate with existing economic_stress variable: {match_rate:.2%}")

In [ ]:
#=========FINAL RESULT========================
#
# MODEL PERFORMANCE TABLE

results_df_sorted = results_df.sort_values(
    by="F1 Score",
    ascending=False
).round(4)

results_df_sorted

In [ ]:
#  MODEL PERFORMANCE COMPARISON

results_long = results_df.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=results_long,
    x="Model",
    y="Score",
    hue="Metric"
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xlabel("Model")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.legend(title="Metric", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# CONFUSION MATRICES FOR ALL MODELS

confusion_results = []
model_predictions = {}

for model, name in models:

    # Train model
    model.fit(X_train, y_train)

    # Predict test data
    y_pred = model.predict(X_test)

    # Save predictions
    model_predictions[name] = y_pred

    # Generate confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    # Extract values
    tn, fp, fn, tp = cm.ravel()

    # Store results
    confusion_results.append({
        "Model": name,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "True Positive": tp,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })

confusion_summary_df = pd.DataFrame(confusion_results)

confusion_summary_df.round(4)

In [ ]:
# PLOT CONFUSION MATRICES

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (model, name) in zip(axes, models):

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No Stress", "Stress"]
    )

    disp.plot(
        cmap="Blues",
        values_format="d",
        ax=ax,
        colorbar=False
    )

    ax.set_title(f"Confusion Matrix - {name}")

plt.tight_layout()
plt.show()

In [ ]:
# NORMALIZED CONFUSION MATRICES FOR ALL MODELS


fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (model, name) in zip(axes, models):

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    cm_normalized = confusion_matrix(
        y_test,
        y_pred,
        normalize="true"
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_normalized,
        display_labels=["No Stress", "Stress"]
    )

    disp.plot(
        cmap="Blues",
        values_format=".2%",
        ax=ax,
        colorbar=False
    )

    ax.set_title(f"Normalized Confusion Matrix - {name}")

plt.tight_layout()
plt.show()

In [ ]:
# CLASSIFICATION REPORTS FOR ALL MODELS

for model, name in models:

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("=" * 70)
    print(f"Classification Report: {name}")
    print("=" * 70)

    print(classification_report(
        y_test,
        y_pred,
        target_names=["No Stress", "Stress"]
    ))

In [ ]:
# ROC CURVES

fig, ax = plt.subplots(figsize=(8, 6))

for model, name in models:
    model.fit(X_train, y_train)
    RocCurveDisplay.from_estimator(
        model,
        X_test,
        y_test,
        name=name,
        ax=ax
    )

plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.title("ROC Curves for Macroeconomic Stress Models")
plt.legend()
plt.show()

In [ ]:
# CROSS VALIDATION
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = []

for model, name in models:

    scores = cross_validate(
        estimator=model,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Mean Accuracy": scores["test_accuracy"].mean(),
        "Std Accuracy": scores["test_accuracy"].std(),
        "Mean Precision": scores["test_precision"].mean(),
        "Std Precision": scores["test_precision"].std(),
        "Mean Recall": scores["test_recall"].mean(),
        "Std Recall": scores["test_recall"].std(),
        "Mean F1": scores["test_f1"].mean(),
        "Std F1": scores["test_f1"].std(),
        "Mean ROC-AUC": scores["test_roc_auc"].mean(),
        "Std ROC-AUC": scores["test_roc_auc"].std()
    })

cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    by="Mean F1",
    ascending=False
).reset_index(drop=True)

cv_results_df.round(4)

In [ ]:
# CLEAN CV TABLE FOR PAPER


cv_table_paper = cv_results_df[
    [
        "Model",
        "Mean Accuracy",
        "Mean Precision",
        "Mean Recall",
        "Mean F1",
        "Mean ROC-AUC",
        "Std F1"
    ]
].round(4)

cv_table_paper

In [ ]:
# FEATURE IMPORTANCE / MODEL INTERPRETATION FOR ALL MODELS

feature_name_map = {
    "gdp_growth_pct": "GDP Growth",
    "gdp_per_capita_current_usd": "GDP per Capita",
    "inflation_pct": "Inflation",
    "unemployment_pct": "Unemployment",
    "current_account_pct_gdp": "Current Account",
    "exports_pct_gdp": "Exports",
    "imports_pct_gdp": "Imports",
    "trade_pct_gdp": "Trade Openness",
    "fdi_net_inflows_pct_gdp": "FDI Inflows",
    "exchange_rate_lcu_per_usd": "Exchange Rate Level",
    "population": "Population",
    "misery_index": "Misery Index",
    "trade_balance_proxy": "Trade Balance Proxy",
    "growth_inflation_gap": "Growth-Inflation Gap",
    "external_pressure_score": "External Pressure Score",
    "exchange_rate_change_pct": "Exchange Rate Change"
}

importance_list = []

for model, name in models:

    # Fit model
    model.fit(X_train, y_train)

    # --------------------------------------------------------
    # Extract final estimator safely
    # --------------------------------------------------------
    if hasattr(model, "named_steps"):
        final_step_name = list(model.named_steps.keys())[-1]
        estimator = model.named_steps[final_step_name]
    else:
        estimator = model

    print(f"{name} final estimator:", estimator.__class__.__name__)

    # --------------------------------------------------------
    # Get importance values
    # --------------------------------------------------------
    if hasattr(estimator, "feature_importances_"):
        importances = estimator.feature_importances_
        importance_type = "Tree-based feature importance"

    elif hasattr(estimator, "coef_"):
        importances = np.abs(estimator.coef_[0])
        importance_type = "Absolute coefficient"

    else:
        print(f"Skipping {name}: no feature importance or coefficient attribute.")
        continue

    # --------------------------------------------------------
    # Build table
    # --------------------------------------------------------
    temp_df = pd.DataFrame({
        "Model": name,
        "Feature": feature_cols,
        "Feature Label": [feature_name_map.get(f, f) for f in feature_cols],
        "Importance": importances,
        "Importance Type": importance_type
    })

    temp_df["Normalized Importance"] = (
        temp_df["Importance"] / temp_df["Importance"].sum()
    )

    importance_list.append(temp_df)

feature_importance_all = pd.concat(importance_list, ignore_index=True)

feature_importance_all.sort_values(
    by=["Model", "Normalized Importance"],
    ascending=[True, False]
).head(20)

In [ ]:
# TOP 5 FEATURES


top5_features_each_model = (
    feature_importance_all
    .sort_values(["Model", "Normalized Importance"], ascending=[True, False])
    .groupby("Model")
    .head(5)
    [["Model", "Feature Label", "Normalized Importance", "Importance Type"]]
)

top5_features_each_model["Normalized Importance"] = (
    top5_features_each_model["Normalized Importance"].round(4)
)

top5_features_each_model

In [ ]:
# AVERAGE FEATURE IMPORTANCE ACROSS ALL MODELS

avg_importance = (
    feature_importance_all
    .groupby("Feature Label")["Normalized Importance"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

top10_avg_importance = avg_importance.head(10)

top10_avg_importance

In [ ]:
# FEATURE IMPORTANCE COMPARISON ACROSS MODELS

top_features = (
    feature_importance_all
    .groupby("Feature Label")["Normalized Importance"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .index
)

importance_top = feature_importance_all[
    feature_importance_all["Feature Label"].isin(top_features)
]

plt.figure(figsize=(12, 7))

sns.barplot(
    data=importance_top,
    x="Normalized Importance",
    y="Feature Label",
    hue="Model"
)

plt.title("Feature Importance Comparison Across Models")
plt.xlabel("Normalized Importance")
plt.ylabel("Feature")
plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()